# AgriTrust — Demand Forecaster Training
Trains an XGBoost (or Random Forest) regressor to predict crop demand by month/season.

**Output:** `ml_weights/demand_model_v1.pkl`

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'data/training'))
print('Repo root:', REPO_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

try:
    from xgboost import XGBRegressor
    XGB = True
    print('XGBoost available')
except ImportError:
    XGB = False
    print('XGBoost not installed — using Random Forest')

## 1. Load Demand Data

In [ ]:
def load_data():
    try:
        from backend.app.db.session import SessionLocal
        from backend.app.models.transaction import Order, OrderStatus
        from backend.app.models.listing import Listing
        db = SessionLocal()
        orders = db.query(Order).filter(Order.status == OrderStatus.COMPLETED).all()
        if len(orders) < 20:
            raise ValueError('Not enough completed orders in DB yet')
        rows = []
        for o in orders:
            if not o.created_at:
                continue
            rows.append({
                'date': o.created_at,
                'crop': o.listing.product_type if o.listing else 'unknown',
                'demand': float(o.quantity or 0),
                'price': float(o.price_per_unit or 0),
                'season': _get_season(o.created_at.month)
            })
        db.close()
        df = pd.DataFrame(rows)
        print(f'Loaded {len(df)} orders from database')
        return df
    except Exception as e:
        print(f'DB unavailable ({e}) — using synthetic data')
        from utils.data_loader import generate_sample_data
        return generate_sample_data('demand')

def _get_season(month):
    if month in [11, 12, 1, 2, 3]:  return 'rainy'
    if month in [4, 5]:             return 'harvest'
    return 'dry'

df = load_data()
df['date'] = pd.to_datetime(df['date'])
df.head()

## 2. Feature Engineering

In [ ]:
features = pd.DataFrame()
features['month']          = df['date'].dt.month
features['quarter']        = df['date'].dt.quarter
features['year']           = df['date'].dt.year
features['price']          = df['price']
features['crop_encoded']   = pd.factorize(df['crop'])[0]
features['season_encoded'] = df['season'].map({'harvest': 0, 'dry': 1, 'rainy': 2}).fillna(1)

target = df['demand']
print(f'Features: {list(features.columns)}')
print(f'Target range: {target.min():.0f} – {target.max():.0f}')

## 3. Train Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

if XGB:
    model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
else:
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

model.fit(X_train, y_train)
print('Training complete')

## 4. Evaluate

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f'MAE: {mae:.1f} units')
print(f'R²:  {r2:.4f}')

plt.figure(figsize=(8, 4))
plt.scatter(y_test, y_pred, alpha=0.4, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Demand')
plt.ylabel('Predicted Demand')
plt.title('Demand Forecaster — Actual vs Predicted')
plt.tight_layout()
plt.show()

## 5. Save

In [ ]:
WEIGHTS = os.path.join(REPO_ROOT, 'ml_weights')
os.makedirs(WEIGHTS, exist_ok=True)
joblib.dump(model,                    f'{WEIGHTS}/demand_model_v1.pkl')
joblib.dump(features.columns.tolist(),f'{WEIGHTS}/demand_features_v1.pkl')
print('Saved to ml_weights/')